In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "021626"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? How does this expression change with dose?

- Model #1: Limma ANOVA (3 group comparison)
    - Control vs. NP1 0.05 vs. NP1 0.1
    - Control vs. NP1 Rhodamine B 0.05 vs. NP1 Rhodamine B 0.1
- Model #2: Limma (2 group comparison)
    - NP1 0.05 vs. NP1 Rhodamine B 0.05
    - NP1 0.1 vs. NP1 Rhodamine B 0.1
    - NP1 0.1 vs. NP2 0.1

In [4]:
# converting to a wide format to run limma
split_protein_df = protein_df %>%
    group_by(Treatment, Dose) %>%
    group_split()

control_df = split_protein_df[[1]]
np1_0.05_df = split_protein_df[[2]]
np1_0.1_df = split_protein_df[[3]]
np1r_0.05_df = split_protein_df[[4]]
np1r_0.1_df = split_protein_df[[5]]
np2_df = split_protein_df[[6]]

wide_matrix = function(split_df, control_df){

    # recombining the control data with all other tx groups
    combined_df = rbind(split_df, control_df)
    
    wide_version = combined_df %>%
        select(Sample_ID, UniProt_ID, Conc_pslog2) %>%
        pivot_wider(names_from = Sample_ID, values_from = Conc_pslog2) %>%
        column_to_rownames("UniProt_ID") %>%
        as.matrix()

    return(wide_version)
}

# calling fn
wide_np1_0.05_matrix = wide_matrix(np1_0.05_df, control_df)
wide_np1_0.1_matrix = wide_matrix(np1_0.1_df, control_df)
wide_np1r_0.05_matrix = wide_matrix(np1r_0.05_df, control_df)
wide_np1r_0.1_matrix = wide_matrix(np1r_0.1_df, control_df)
wide_np2_matrix = wide_matrix(np2_df, control_df)

head(wide_np1_0.05_matrix)

,NP1_6_0.05,NP1_14_0.05,NP1_24_0.05,Control_2_NA,Control_10_NA,Control_17_NA
Q07011,1.572613,1.554672,1.576927,1.585350,1.583870,1.575114
Q15109,2.143017,2.148347,2.139980,2.135693,2.073869,2.138415
Q9UNG2,1.646805,1.650657,1.660484,1.642756,1.674437,1.629017
P31749,1.663189,1.667402,1.684437,1.670583,1.663760,1.649666
Q15389,1.803776,1.791555,1.810487,1.896098,1.877910,1.851314
P05089,2.109544,2.149856,2.132945,2.120742,2.113760,2.121512


In [5]:
# creating a sample info df
create_sample_info = function(wide_matrix, dose, Variable){

    sample_info_df = protein_df[,2:4] %>%
        unique() %>%
        filter(Treatment == 'Control' | Dose == dose & Treatment == Variable) %>%
        # ensuring the sample ids are in the same order as the previous df
        arrange(match(Sample_ID, colnames(wide_matrix)))

    return(sample_info_df)
    
    }

# calling fn
np1_0.05_sample_df = create_sample_info(wide_np1_0.05_matrix, 0.05, "NP1")
np1_0.1_sample_df = create_sample_info(wide_np1_0.1_matrix, 0.1, "NP1")
np1r_0.05_sample_df = create_sample_info(wide_np1r_0.05_matrix, 0.05, "NP1-Rhodamine B")
np1r_0.1_sample_df = create_sample_info(wide_np1r_0.1_matrix, 0.1, "NP1-Rhodamine B")
np2_sample_df = create_sample_info(wide_np2_matrix, 0.1, "NP2")

head(np1_0.05_sample_df)

,Treatment,Sample_ID,Dose
,<chr>,<chr>,<dbl>
1,NP1,NP1_6_0.05,0.05
2,NP1,NP1_14_0.05,0.05
3,NP1,NP1_24_0.05,0.05
4,Control,Control_2_NA,0.00
5,Control,Control_10_NA,0.00
6,Control,Control_17_NA,0.00


Running two group limma models.

In [6]:
get_limma_results = function(model, sample_df, wide_matrix, comparison){
    
    # running limma
    # comparing each tx group to the control
    design = model.matrix(as.formula(paste0("~", model)), data = sample_df)
    linear_fit = lmFit(wide_matrix, design)
    limma_fit = eBayes(linear_fit, robust = TRUE)
    
    treatments = colnames(design)[2:length(colnames(design))]
    limma_df = data.frame()
    for (i in 1:length(treatments)){
    
        # limma's defaults gives us a BH adjusted p value
        limma_results = topTable(limma_fit, coef = treatments[i], number = Inf) %>%
            rownames_to_column("UniProt_ID") %>%
            mutate(Comparison = comparison, Dose = unique(sample_df$Dose)[1]) 
    
        limma_df = rbind(limma_df, limma_results)
    
        }
    
    # cleaning up the df
    limma_df = limma_df[,c(1,2,5,6,8,9)] %>%
        # convert log fc to log2 fc
        mutate(log2FC = log2(10^logFC))
    colnames(limma_df)[3:4] = c("P Value", "P Adj")
    
    # adding in meta data
    final_df = inner_join(limma_df, unique(protein_df[,6:8]))[,c(8,1,9,5,7,3,4)]

    return(final_df)
    }

# recombining dfs
np1_np1r_0.05_sample_df = rbind(np1_0.05_sample_df[1:3,], np1r_0.05_sample_df[1:3,])
matrix_np1_npr1_0.05 = cbind(wide_np1_0.05_matrix[,1:3],wide_np1r_0.05_matrix[,1:3])
np1_np1r_0.1_sample_df = rbind(np1_0.1_sample_df[1:3,], np1r_0.1_sample_df[1:3,])
matrix_np1_npr1_0.1 = cbind(wide_np1_0.1_matrix[,1:3],wide_np1r_0.1_matrix[,1:3])
np1_np2_0.1_sample_df = rbind(np1_0.1_sample_df[1:3,], np2_sample_df[1:3,])
matrix_np1_np2 = cbind(wide_np1_0.1_matrix[,1:3],wide_np2_matrix[,1:3])

# calling fn
np1_0.05_results_df = get_limma_results('Treatment', np1_np1r_0.05_sample_df, matrix_np1_npr1_0.05, "NP1 vs. NP1-Rhodamine B 0.05mg/mL")
np1_0.1_results_df = get_limma_results('Treatment', np1_np1r_0.1_sample_df, matrix_np1_npr1_0.1, "NP1 vs. NP1-Rhodamine B 0.1mg/mL")
np1_np2_results_df = get_limma_results('Treatment', np1_np2_0.1_sample_df, matrix_np1_np2, "NP1 vs. NP2 0.1mg/mL")

Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`


In [7]:
# combining for export
two_group_comparison_df = rbind(np1_0.05_results_df, np1_0.1_results_df, np1_np2_results_df)
head(two_group_comparison_df)

,ELISA_ID,UniProt_ID,Protein_Name,Comparison,log2FC,P Value,P Adj
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,nEL04021,O00300,TNFRSF11B,NP1 vs. NP1-Rhodamine B 0.05mg/mL,0.6217117,6.180248e-10,1.730469e-07
2,nEL03731,P10909,CLU,NP1 vs. NP1-Rhodamine B 0.05mg/mL,0.3957803,4.457688e-06,6.240764e-04
3,nEL03981,P10646,TFPI,NP1 vs. NP1-Rhodamine B 0.05mg/mL,-0.2101744,7.870113e-05,7.132558e-03
4,nEL00061,P10147,CCL3,NP1 vs. NP1-Rhodamine B 0.05mg/mL,-0.2433688,1.018937e-04,7.132558e-03
5,nEL00351,P42830,CXCL5,NP1 vs. NP1-Rhodamine B 0.05mg/mL,0.2141613,1.037832e-03,5.723293e-02
6,nEL00501,P09919,G-CSF,NP1 vs. NP1-Rhodamine B 0.05mg/mL,0.1297393,1.226420e-03,5.723293e-02


Running three group limma models.

In [8]:
# reformatting dfs for models
np1_sample_df = rbind(np1_0.05_sample_df, np1_0.1_sample_df[1:3,]) %>%
    mutate(Exposure = ifelse(Dose == 0.1, "NP1_High", 
                             ifelse(Dose == 0.05, "NP1_Low", Treatment)))
np1r_sample_df = rbind(np1r_0.05_sample_df, np1r_0.1_sample_df[1:3,]) %>%
    mutate(Exposure = ifelse(Dose == 0.1, "NP1r_High", 
                             ifelse(Dose == 0.05, "NP1r_Low", Treatment)))
wide_np1_matrix = cbind(wide_np1_0.05_matrix, wide_np1_0.1_matrix[,1:3])
wide_np1r_matrix = cbind(wide_np1r_0.05_matrix, wide_np1r_0.1_matrix[,1:3])

# the 0 + means “estimate group means directly”.
np1_design = model.matrix(~ 0 + Exposure, data = np1_sample_df)
np1r_design = model.matrix(~ 0 + Exposure, data = np1r_sample_df)

np1_contrast_matrix = makeContrasts(
  Low_vs_Control  = ExposureNP1_Low  - ExposureControl,
  High_vs_Control = ExposureNP1_High - ExposureControl,
  High_vs_Low     = ExposureNP1_High - ExposureNP1_Low,
  levels = np1_design
)
np1r_contrast_matrix = makeContrasts(
  Low_vs_Control  = ExposureNP1r_Low  - ExposureControl,
  High_vs_Control = ExposureNP1r_High - ExposureControl,
  High_vs_Low     = ExposureNP1r_High - ExposureNP1r_Low,
  levels = np1r_design
)

In [9]:
get_anova_limma_results = function(sample_df, wide_matrix, design, contrast_matrix, comparison){
    # ADD WORDS

    fit = lmFit(wide_matrix, design)
    fit = contrasts.fit(fit, contrast_matrix)
    limma_fit = eBayes(fit)

    treatments = colnames(limma_fit$coefficients)
    limma_df = data.frame()
    for (i in 1:length(treatments)){
    
        # limma's defaults gives us a BH adjusted p value
        limma_results = topTable(limma_fit, coef = treatments[i], number = Inf) %>%
            rownames_to_column("UniProt_ID") %>%
            mutate(Comparison = comparison, Exposure = treatments[i], Dose = unique(sample_df$Dose)[1]) 
    
        limma_df = rbind(limma_df, limma_results)
    
        }
    
    # cleaning up the df
    limma_df = limma_df[,c(1,2,5,6,8,9)] %>%
        mutate(Exposure = ifelse(grepl("Exposure", Exposure), 
                        unlist(strsplit(Exposure, split = "Exposure"))[2], Exposure)) %>%
        # convert log fc to log2 fc
        mutate(log2FC = log2(10^logFC))
    colnames(limma_df)[3:4] = c("P Value", "P Adj")
    
    # adding in meta data
    final_df = inner_join(limma_df, unique(protein_df[,6:8]))[,c(8,1,9,5:7,3,4)]

    return(final_df)
    }

# calling fn
anova_np1_results_df = get_anova_limma_results(np1_sample_df, wide_np1_matrix, np1_design, np1_contrast_matrix, 
                                               "Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL")
anova_np1r_results_df = get_anova_limma_results(np1r_sample_df, wide_np1r_matrix, np1r_design, np1r_contrast_matrix, 
                                               "Control vs. NP1-Rhodamine B 0.05mg/mL vs. NP1-Rhodamine B 0.1mg/mL")

Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`


In [10]:
# combining for export
three_group_comparison_df = rbind(anova_np1_results_df, anova_np1r_results_df)
head(three_group_comparison_df)

,ELISA_ID,UniProt_ID,Protein_Name,Comparison,Exposure,log2FC,P Value,P Adj
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,nEL03731,P10909,CLU,Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,-1.1552479,1.976759e-12,5.534927e-10
2,nEL04021,O00300,TNFRSF11B,Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,-1.0388131,3.511780e-10,4.916492e-08
3,nEL00061,P10147,CCL3,Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,0.4542374,2.856369e-08,2.665944e-06
4,nEL08971,Q99538,LGMN,Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,-0.2389893,5.691948e-05,3.984363e-03
5,nEL03101,Q15389,Angiopoietin-1,Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,-0.2430584,7.386339e-05,4.136350e-03
6,nEL01981,Q99988,GDF-15 (MIC-1),Control vs. NP1 0.05mg/mL vs. NP1 0.1mg/mL,Low_vs_Control,0.2398895,1.030797e-04,4.810385e-03


In [11]:
# exporting
write.xlsx(two_group_comparison_df, paste0(Output,"/", "Two_Group_Results_", cur_date, ".xlsx"), 
           rowNames = FALSE)
write.xlsx(three_group_comparison_df, paste0(Output,"/", "Three_Group_Results_", cur_date, ".xlsx"), 
           rowNames = FALSE)

We have a couple of design options for the limma model:

- If we wanted to control for dose, it would help us determine treatment's effect *regardless* of dose (ie. Treatment + Dose). Usually this a covariate that you're trying to minimize the impact on your date to isolate your research question.
- If we we're also interested in determining how treatment and protein expression are impacted by dose, we'd use: Treatment + Dose. However, we can't run an interaction for a dependent variable with only one dose, so to keep it consistent across all treatments I'll just stratify to see how dose modifies protein expression too. Additionally, all the controls have a dose of zero, which would introduce bias and doesn't allow use to use a interaction here.

Therefore, I'll just stratify, despite reducing statistical power.